# IM 主力合约测试

测试 IM 主力合约的筛选逻辑和交易信号生成

**本文件用于生成模拟仓位数据（15 分钟频率信号），供 trade_backtest.ipynb 回测框架使用**

### 文件结构说明
1. 数据加载：读取 1 分钟期货行情数据
2. 主力合约筛选：每日选择 Code 最大的 IM 合约
3. 信号生成：`generate_position_signals()` 函数生成指定频率的仓位信号
4. 回测调用：将生成的信号传入 trade_backtest.ipynb 运行回测

In [ ]:
import pandas as pd
import numpy as np

## 1. 数据加载

加载期货 1 分钟行情数据，包含以下字段：
- `date`: 交易日期
- `minTime`: 分钟时间 (HHMM 格式)
- `Code`: 合约代码
- `twap`: 时间加权平均价格

In [ ]:
# 加载行情数据
price_data = pd.read_parquet('/home/dev1/future_min1/20200102_20250313_bar.parquet')[['twap']].reset_index()
print("行情数据形状:", price_data.shape)
print("日期范围:", price_data['date'].min(), "-", price_data['date'].max())

## 2. IM 合约筛选

筛选所有 IM（中证 1000 股指期货）合约数据

In [ ]:
# 筛选 IM 合约
im_data = price_data[price_data['Code'].str.startswith('IM')].copy()
print("\nIM 合约数据形状:", im_data.shape)
print("IM 合约日期范围:", im_data['date'].min(), "-", im_data['date'].max())
print("IM 合约数量:", im_data['Code'].nunique())

## 3. 主力合约筛选

**主力合约筛选逻辑：**
- 每日选择 Code 数值最大的合约作为主力合约
- 例如：IM2401、IM2402、IM2403 中，IM2403 是主力合约
- 这是因为期货合约代码数字越大，通常代表越远月的合约，流动性更好

In [ ]:
# 筛选每日 IM 主力合约（Code 最大的合约）
def select_main_contract(df):
    """筛选每日主力合约（Code 最大值）"""
    return df.groupby('date', group_keys=False).apply(
        lambda x: x[x['Code'] == x['Code'].max()]
    ).reset_index(drop=True)

main_contract_data = select_main_contract(im_data)
print("\n主力合约数据形状:", main_contract_data.shape)
print("交易日数量:", main_contract_data['date'].nunique())
print("\n主力合约示例:")
main_contract_data.head(10)

In [ ]:
# 查看主力合约换月情况
main_contracts_by_date = main_contract_data.groupby('date')['Code'].first()
print("\n主力合约换月情况:")
print(main_contracts_by_date.head(30))

In [ ]:
# 统计每个主力合约的交易日数
contract_counts = main_contracts_by_date.value_counts()
print("\n主力合约交易日统计:")
print(contract_counts)

## 4. 生成 15 分钟频率仓位信号

**此函数用于生成模拟仓位数据，只在生成模拟仓位时才使用 15 分钟频率**

### 信号生成逻辑
- **起始时间**: 09:46（586 分钟），跳过早盘前 15 分钟
- **信号频率**: 每隔 `interval` 分钟生成一个信号（默认 15 分钟）
- **信号方向**: 交替生成多头 (1) 和空头 (-1) 信号
  - 第 1 个信号：多头 (direction=1)
  - 第 2 个信号：空头 (direction=-1)
  - 第 3 个信号：多头 (direction=1)
  - ...
- **信号手数**: 每次 1 手

### 信号含义
- `direction=1` (多头信号): 平空仓，买入平仓
- `direction=-1` (空头信号): 平多仓，卖出平仓

### 自定义频率
可以通过 `interval` 参数调整信号频率：
- `interval=5`: 每 5 分钟一个信号
- `interval=15`: 每 15 分钟一个信号
- `interval=30`: 每 30 分钟一个信号
- `interval=60`: 每 60 分钟一个信号

In [ ]:
def generate_position_signals(price_data: pd.DataFrame, interval: int = 15) -> pd.DataFrame:
    """
    生成模拟仓位信号数据
    
    参数:
    - price_data: pd.DataFrame, 行情数据（主力合约数据）
    - interval: int, 信号频率（分钟），默认 15 分钟
    
    返回:
    - position_df: pd.DataFrame, 仓位信号
      columns: ['date', 'minTime', 'Code', 'direction', 'numbers']
    
    信号规则:
    - 从 09:46 开始（586 分钟），每隔 interval 分钟生成一个信号
    - 交替生成多头和空头信号
    - 每次 1 手
    """
    # 构建时间索引
    time_index = {}
    for (date, code), group in price_data.groupby(['date', 'Code']):
        times = sorted(group['minTime'].unique())
        time_index[(date, code)] = times
    
    all_signals = []
    
    for (date, code), times in time_index.items():
        signals = []
        
        for time in times:
            # 转换为分钟数
            hour = int(time[:2])
            minute = int(time[2:])
            total_minutes = hour * 60 + minute
            
            # 从 09:46 开始（586 分钟）
            if total_minutes < 586:
                continue
            
            # 每隔 interval 分钟生成一个信号
            if (total_minutes - 586) % interval == 0:
                direction = 1 if (len(signals) % 2 == 0) else -1
                signals.append({
                    'date': date,
                    'minTime': time,
                    'Code': code,
                    'direction': direction,
                    'numbers': 1
                })
        
        all_signals.extend(signals)
    
    return pd.DataFrame(all_signals)


# 生成 15 分钟频率的仓位信号
signals_15min = generate_position_signals(main_contract_data, interval=15)
print(f"\n15 分钟信号数量：{len(signals_15min)}")
print("信号示例:")
signals_15min.head(10)

## 5. 信号测试

验证生成的信号是否符合预期

In [ ]:
# 测试第一个交易日的信号
test_date = main_contract_data['date'].unique()[0]
test_signals = signals_15min[signals_15min['date'] == test_date]
print(f"\n日期 {test_date} 的信号:")
print(f"信号数量：{len(test_signals)}")
print(test_signals)

## 6. 使用生成的信号运行回测

将生成的仓位信号传入 trade_backtest.ipynb 中的 PositionBacktester 运行回测

### 使用步骤
1. 在 trade_backtest.ipynb 中导入此文件的 `generate_position_signals` 函数
2. 生成指定频率的信号：`signals_df = generate_position_signals(main_contract_data, interval=15)`
3. 运行回测：`trade_records, daily_stats = backtester.run_backtest(signals_df)`
4. 查看绩效指标：`metrics = backtester.calculate_performance_metrics()`

### 示例代码
```python
# 在 trade_backtest.ipynb 中
from test import generate_position_signals

# 生成 15 分钟频率信号
signals_df = generate_position_signals(main_contract_data, interval=15)

# 运行回测
trade_records, daily_stats = backtester.run_backtest(signals_df)

# 计算绩效
metrics = backtester.calculate_performance_metrics()
```

In [ ]:
# 示例：使用生成的信号运行回测
# 注意：需要先运行 trade_backtest.ipynb 初始化回测引擎

# 保存信号数据（可选）
# signals_15min.to_feather('position_signals_15min.feather')
# print("信号数据已保存至 position_signals_15min.feather")

# 在 trade_backtest.ipynb 中使用:
# from test import generate_position_signals
# signals_df = generate_position_signals(main_contract_data, interval=15)
# trade_records, daily_stats = backtester.run_backtest(signals_df)

print("\n使用方法:")
print("1. 在 trade_backtest.ipynb 中导入此文件的 generate_position_signals 函数")
print("2. 生成信号：signals_df = generate_position_signals(main_contract_data, interval=15)")
print("3. 运行回测：trade_records, daily_stats = backtester.run_backtest(signals_df)")